# Questão 03: Pós-Treino com LoRA/QLoRA (+ Bônus Qwen2.5-1.5B)

Repetição do experimento da Questão 02 com QLoRA (base em 4 bits, adaptadores de baixo posto), experimento bônus com o Qwen2.5-1.5B e a consolidação final das quatro rodadas. Contém o *setup* compartilhado (Seção 0). Saídas preservadas da execução validada no Kaggle.


## Seção 0: Setup global

A célula abaixo precisa ser a primeira célula de código do notebook: `CUDA_VISIBLE_DEVICES` limita a sessão a uma única GPU antes de qualquer `import torch`. Com as duas T4 visíveis, o Trainer ativaria DataParallel e o gather dos logits (vocabulário de 151k tokens) estoura a memória da GPU 0. O modelo de 0.5B cabe com folga em uma única T4.

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import time
import random

MODO_TESTE = False
RODAR_Q1 = True
RODAR_BONUS_15B = True

SEED = 42
MAX_LEN = 256
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
MODEL_NAME_BONUS = 'Qwen/Qwen2.5-1.5B'
T0 = time.time()

LIMITE_BONUS_H = 9.5
LIMITE_TETO_H = 11.0

if MODO_TESTE:
    Q1_MAX_DOCS = 300
    Q1_N_AVALIACAO = 50
    Q1_EPOCAS = 1
    Q2_ALVO_PARES = 120
    Q2_EPOCAS = 1
    Q3_EPOCAS = 1
    BONUS_MAX_STEPS = 10
    N_BENCHMARK_Q1 = 5
else:
    Q1_MAX_DOCS = None
    Q1_N_AVALIACAO = 1000
    Q1_EPOCAS = 2
    Q2_ALVO_PARES = 2000
    Q2_EPOCAS = 3
    Q3_EPOCAS = 3
    BONUS_MAX_STEPS = -1
    N_BENCHMARK_Q1 = 25


def tempo_decorrido_h():
    return (time.time() - T0) / 3600


random.seed(SEED)
print(f'MODO_TESTE={MODO_TESTE} | RODAR_Q1={RODAR_Q1} | RODAR_BONUS_15B={RODAR_BONUS_15B}')
print(f'Modelo principal: {MODEL_NAME} | MAX_LEN={MAX_LEN} | SEED={SEED}')

### Instalação segura de dependências

A stack do Kaggle (torch, transformers 5.x, datasets, accelerate) fica intocada: reinstalar qualquer um desses pacotes foi o que quebrou os imports nas primeiras tentativas da Questão 01. Só `peft` e `bitsandbytes` são instalados, com `--no-deps`, e apenas se estiverem ausentes. `peft` ausente é erro fatal imediato (a Questão 03 depende dele); `bitsandbytes` ausente apenas desativa o caminho QLoRA (o LoRA puro cumpre a questão).

In [ ]:
import importlib.util
import subprocess
import sys


def garantir_pacote(nome):
    if importlib.util.find_spec(nome) is None:
        print(f'Instalando {nome} com --no-deps...')
        resultado = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', nome],
            capture_output=True, text=True)
        if resultado.returncode != 0:
            print(resultado.stderr[-800:])
        importlib.invalidate_caches()


garantir_pacote('peft')
garantir_pacote('bitsandbytes')

PEFT_OK = False
BNB_OK = False
try:
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    import peft
    PEFT_OK = True
except Exception as erro:
    print(f'peft indisponível: {erro}')
try:
    import bitsandbytes
    BNB_OK = True
except Exception as erro:
    print(f'bitsandbytes indisponível: {erro}')

if not PEFT_OK:
    raise RuntimeError('peft é obrigatório para a Questão 3 e não pôde ser carregado. Verifique se a Internet do notebook está ligada.')

import torch
import transformers

print(f'torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}')
print(f'PEFT_OK={PEFT_OK} | BNB_OK={BNB_OK}')

In [ ]:
import gc
import json
import math
import re
import shutil
import hashlib
import traceback
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
    default_data_collator,
)

try:
    from transformers import BitsAndBytesConfig
except Exception as erro:
    print(f'BitsAndBytesConfig indisponível: {erro}')
    BNB_OK = False

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    assert torch.cuda.device_count() == 1, 'Esperada exatamente 1 GPU visível'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'Disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')

### Funções compartilhadas

Definidas uma única vez e usadas pelas três questões: salvamento de artefatos, estilo e helpers de gráfico, montagem de prompts no formato Alpaca e limpeza de memória entre seções.

In [ ]:
DIR_FIGURAS = 'figuras'
os.makedirs(DIR_FIGURAS, exist_ok=True)

COR_ANTES = '#b5651d'
COR_DEPOIS = '#12805a'
C_FULL = '#b5651d'
C_LORA = '#2a78d6'
C_QLORA = '#12805a'
C_BONUS = '#4a3aa7'
C_BASE = '#898781'
C_EDA = '#2a78d6'
COR_GRADE = '#e1e0d9'

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
    'axes.grid': False,
    'axes.axisbelow': True,
})

RESULTADOS = {}


def salvar_json(objeto, caminho):
    conteudo = json.dumps(objeto, ensure_ascii=False, indent=2, default=str)
    with open(caminho, 'w', encoding='utf-8') as arquivo:
        arquivo.write(conteudo)
    print(f'Salvo: {caminho}')
    if len(conteudo) < 100000:
        print(f'----- backup em log: inicio de {caminho} -----')
        print(conteudo)
        print(f'----- backup em log: fim de {caminho} -----')


def salvar_fig(fig, nome):
    caminho = os.path.join(DIR_FIGURAS, nome)
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Figura salva: {caminho}')


def rotular_barras(ax, barras, sufixo='', casas=2):
    for barra in barras:
        altura = barra.get_height()
        ax.annotate(f'{altura:.{casas}f}{sufixo}',
                    xy=(barra.get_x() + barra.get_width() / 2, altura),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')


def grafico_metricas_multi(series, titulo, nome_arquivo):
    paineis = [('Entropia Cruzada', 'entropia_cruzada', '', 3),
               ('Perplexidade', 'perplexidade', '', 2),
               ('Acurácia de Tokens (%)', 'acuracia_tokens', '%', 1)]
    largura = 12 + max(0, len(series) - 2) * 1.4
    fig, eixos = plt.subplots(1, 3, figsize=(largura, 4.2))
    for ax, (nome, chave, sufixo, casas) in zip(eixos, paineis):
        rotulos = [s[0] for s in series]
        valores = [s[1][chave] for s in series]
        cores = [s[2] for s in series]
        barras = ax.bar(rotulos, valores, color=cores, width=0.62)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
        if len(series) > 2:
            ax.tick_params(axis='x', labelrotation=20)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_barh(itens, titulo, nome_arquivo, cor=None, log_x=False, fmt='{:,.0f}'):
    nomes = [str(item[0]) for item in itens]
    valores = [item[1] for item in itens]
    fig, ax = plt.subplots(figsize=(9.5, max(3, 0.42 * len(itens) + 1.2)))
    barras = ax.barh(nomes[::-1], valores[::-1], color=cor or C_EDA)
    if log_x:
        ax.set_xscale('log')
    for barra in barras:
        largura = barra.get_width()
        ax.annotate(fmt.format(largura),
                    xy=(largura, barra.get_y() + barra.get_height() / 2),
                    xytext=(4, 0), textcoords='offset points',
                    va='center', fontsize=9)
    ax.set_title(titulo)
    ax.grid(axis='x', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_hist_comprimentos(comprimentos, cortes, titulo, nome_arquivo, mediana=None):
    dados = [c for c in comprimentos if c > 0]
    bins = np.logspace(np.log10(max(min(dados), 1)), np.log10(max(dados)), 50)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(dados, bins=bins, color=C_EDA, edgecolor='white', linewidth=0.3)
    ax.set_xscale('log')
    for corte in cortes:
        ax.axvline(corte, color='#8a1d1d', linestyle='--', linewidth=1.2)
    if mediana:
        ax.axvline(mediana, color='#4a3aa7', linestyle=':', linewidth=1.4)
        ax.annotate(f'mediana {mediana:,.0f}', xy=(mediana, ax.get_ylim()[1] * 0.9),
                    xytext=(6, 0), textcoords='offset points', fontsize=9, color='#4a3aa7')
    ax.set_xlabel('Comprimento (caracteres, escala log)')
    ax.set_ylabel('Documentos')
    ax.set_title(titulo)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curva_loss(log_history, titulo, nome_arquivo):
    pontos_treino = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
    pontos_eval = [(h['step'], h['eval_loss']) for h in log_history if 'eval_loss' in h]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    if pontos_treino:
        xs, ys = zip(*pontos_treino)
        ax.plot(xs, ys, color=C_EDA, linewidth=1.6, label='Loss de treino')
    if pontos_eval:
        xs, ys = zip(*pontos_eval)
        ax.plot(xs, ys, color='#184f95', marker='o', linestyle='--', linewidth=1.2, label='Loss de avaliação')
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curvas_loss(series, titulo, nome_arquivo):
    fig, ax = plt.subplots(figsize=(9, 4.2))
    for rotulo, log_history, cor in series:
        pontos = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
        if pontos:
            xs, ys = zip(*pontos)
            ax.plot(xs, ys, color=cor, linewidth=1.6, label=rotulo)
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss de treino')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_painel_comparativo(entidades, titulo, nome_arquivo):
    paineis = [('Perplexidade (depois)', 'ppl', '', 2),
               ('Acurácia de Tokens (depois, %)', 'acc', '%', 1),
               ('Tempo de treino (min)', 'tempo_min', '', 1),
               ('Pico de VRAM (GB)', 'vram', '', 2)]
    fig, eixos = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (nome, chave, sufixo, casas) in zip(eixos.flat, paineis):
        rotulos = [e['rotulo'] for e in entidades]
        valores = [e[chave] for e in entidades]
        cores = [e['cor'] for e in entidades]
        barras = ax.bar(rotulos, valores, color=cores, width=0.6)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.tick_params(axis='x', labelrotation=15)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def montar_prompt(instrucao, entrada=''):
    if entrada and entrada.strip():
        return f'### Instrução:\n{instrucao}\n\n### Entrada:\n{entrada}\n\n### Resposta:\n'
    return f'### Instrução:\n{instrucao}\n\n### Resposta:\n'


def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto.lower())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]', '', texto)


def taxa_acerto_por_categoria(resultados):
    por_categoria = {}
    for r in resultados:
        acertou = normalizar_texto(r['resposta_esperada']) in normalizar_texto(r['resposta_gerada'])
        ok, total = por_categoria.get(r['cat'], (0, 0))
        por_categoria[r['cat']] = (ok + (1 if acertou else 0), total + 1)
    return {c: round(ok / total * 100, 1) for c, (ok, total) in por_categoria.items()}


def atualizar_zip_parcial():
    os.makedirs('parciais', exist_ok=True)
    for nome in os.listdir('.'):
        if nome.endswith('.json'):
            shutil.copy(nome, os.path.join('parciais', nome))
    if os.path.isdir(DIR_FIGURAS):
        shutil.copytree(DIR_FIGURAS, os.path.join('parciais', DIR_FIGURAS), dirs_exist_ok=True)
    shutil.make_archive('resultados_parciais', 'zip', 'parciais')
    print('Zip parcial atualizado: resultados_parciais.zip')


print('Helpers de artefatos e gráficos definidos.')

In [ ]:
def carregar_modelo_fp32(nome):
    try:
        modelo = AutoModelForCausalLM.from_pretrained(nome, dtype=torch.float32)
    except TypeError:
        modelo = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=torch.float32)
    return modelo.to(DEVICE)


def montar_training_args(**kwargs):
    for tentativa in range(6):
        try:
            return TrainingArguments(**kwargs)
        except (TypeError, ValueError) as erro:
            mensagem = str(erro)
            removido = None
            for chave in list(kwargs):
                if chave in mensagem:
                    removido = chave
                    kwargs.pop(chave)
                    break
            if removido is None:
                raise
            print(f'Argumento removido por incompatibilidade de versão: {removido}')
    raise RuntimeError('Não foi possível montar TrainingArguments')


def calcular_metricas(modelo, tok, textos, prompts=None, batch_size=8):
    modelo.eval()
    total_loss = 0.0
    total_validos = 0
    total_certos = 0
    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio:inicio + batch_size]
        enc = tok(lote, return_tensors='pt', max_length=MAX_LEN, truncation=True, padding=True)
        input_ids = enc['input_ids'].to(DEVICE)
        attention_mask = enc['attention_mask'].to(DEVICE)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        if prompts is not None:
            for j, prompt in enumerate(prompts[inicio:inicio + batch_size]):
                n_prompt = len(tok(prompt)['input_ids'])
                labels[j, :min(n_prompt, labels.shape[1])] = -100
        with torch.no_grad():
            saida = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        mascara_validos = labels[:, 1:] != -100
        n_validos = mascara_validos.sum().item()
        if n_validos == 0:
            continue
        total_loss += saida.loss.item() * n_validos
        total_validos += n_validos
        predicoes = saida.logits.argmax(dim=-1)
        certos = ((predicoes[:, :-1] == input_ids[:, 1:]) & mascara_validos).sum().item()
        total_certos += certos
    entropia = total_loss / max(total_validos, 1)
    return {'entropia_cruzada': round(entropia, 4),
            'perplexidade': round(math.exp(min(entropia, 20)), 2),
            'acuracia_tokens': round(total_certos / max(total_validos, 1) * 100, 2)}


class SFTDatasetMasked(Dataset):
    def __init__(self, pares, tok):
        self.exemplos = []
        for prompt, completo in pares:
            ids_prompt = tok(prompt)['input_ids']
            ids_completo = tok(completo)['input_ids']
            ids_resposta = ids_completo[len(ids_prompt):]
            espaco_resposta = MAX_LEN - len(ids_prompt)
            if espaco_resposta < 2:
                continue
            if len(ids_resposta) > espaco_resposta:
                ids_resposta = ids_resposta[:espaco_resposta - 1] + [tok.eos_token_id]
            input_ids = ids_prompt + ids_resposta
            n_pad = MAX_LEN - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * n_pad
            labels = [-100] * len(ids_prompt) + list(ids_resposta) + [-100] * n_pad
            input_ids = input_ids + [tok.pad_token_id] * n_pad
            self.exemplos.append({
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'labels': torch.tensor(labels),
            })

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, indice):
        return self.exemplos[indice]


def gerar_resposta(modelo, tok, instrucao, entrada='', max_new_tokens=90):
    modelo.eval()
    prompt = montar_prompt(instrucao, entrada)
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def gerar_texto_livre(modelo, tok, prompt, max_new_tokens=80):
    modelo.eval()
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=True,
                                temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                                pad_token_id=tok.eos_token_id)
    return tok.decode(saida[0], skip_special_tokens=True)


def rodar_benchmark(modelo, tok, itens, max_new_tokens=100):
    modelo.eval()
    resultados = []
    for item in itens:
        entradas = tok(item['pergunta'], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.pad_token_id or tok.eos_token_id)
        gerada = tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        resultados.append({'id': item['id'], 'cat': item['cat'], 'pergunta': item['pergunta'],
                           'resposta_esperada': item['resposta'], 'resposta_gerada': gerada})
    return resultados


def responder_qualitativa(modelo, tok, itens):
    respostas = []
    for item in itens:
        resposta = gerar_resposta(modelo, tok, item['instruction'], item.get('input', ''))
        respostas.append({'id': item['id'], 'resposta': resposta})
    return respostas


def liberar_gpu(*nomes):
    espaco = globals()
    for nome in nomes:
        espaco.pop(nome, None)
    gc.collect()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f'VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB | reservada: {torch.cuda.memory_reserved() / 1e9:.2f} GB')


def status_recursos(rotulo):
    print(f'[{rotulo}] tempo decorrido: {tempo_decorrido_h():.2f}h | disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')
    if torch.cuda.is_available():
        print(f'[{rotulo}] VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')


def remover_checkpoints(diretorio):
    if os.path.isdir(diretorio):
        shutil.rmtree(diretorio, ignore_errors=True)
        print(f'Checkpoints removidos: {diretorio}')


print('Helpers de modelo, métricas e memória definidos.')

### Pré-voo

Valida em poucos minutos tudo que poderia falhar depois de horas de treino: acesso aos dois datasets do Hugging Face com os campos esperados, tokenizer, GPU única e disco. Se algo estiver errado, o notebook morre aqui, no minuto 3, e não na hora 6.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: {MODEL_NAME} | vocabulário {tokenizer.vocab_size:,}')

verificacao_docentes = load_dataset('vickminari/docentesDC', split='train')
assert 'text' in verificacao_docentes.column_names, 'Campo text ausente no docentesDC'
assert 'nome_professor' in verificacao_docentes.column_names, 'Campo nome_professor ausente no docentesDC'
print(f'docentesDC ok: {len(verificacao_docentes):,} linhas | colunas {verificacao_docentes.column_names}')
del verificacao_docentes

verificacao_dompi = load_dataset('gutoportelaa/DOMPI-2025', name='raw', split='entre_rios')
assert 'texto' in verificacao_dompi.column_names, 'Campo texto ausente no DOMPI-2025'
print(f'DOMPI-2025 ok (split entre_rios): {len(verificacao_dompi):,} documentos')
del verificacao_dompi
gc.collect()

livre_gb = shutil.disk_usage('.').free / 1e9
if os.path.isdir('/kaggle'):
    assert livre_gb > 15, f'Disco livre insuficiente: {livre_gb:.1f} GB'

status_recursos('pré-voo concluído')

## Seção 3: Questão 03, pós-treino com LoRA e QLoRA

Repete o experimento da Questão 02 com adaptadores de baixo posto, usando exatamente os mesmos dados, split, máscara de loss e função de avaliação. Cada experimento carrega a base do zero (fp32 para LoRA puro, NF4 4-bit para QLoRA), avalia o "antes" AINDA SEM os adaptadores (o PEFT modifica o modelo in-place, então medir depois do `get_peft_model` contaminaria a comparação), treina e avalia o "depois". As métricas do "antes" quantizado vs fp32 isolam o efeito da quantização de graça.

No caminho LoRA puro, `enable_input_require_grads()` é obrigatório com gradient checkpointing (sem ele o backward falha com "element 0 of tensors does not require grad"). No QLoRA, o `prepare_model_for_kbit_training` cumpre esse papel.

In [ ]:
if tempo_decorrido_h() > 10.5:
    Q3_EPOCAS = 2
    print(f'Gate de tempo: {tempo_decorrido_h():.1f}h decorridas, Questão 3 reduzida para 2 épocas')


class TetoDeTempo(TrainerCallback):
    def __init__(self, limite_h):
        self.limite_h = limite_h

    def on_step_end(self, args, state, control, **kwargs):
        if tempo_decorrido_h() > self.limite_h:
            print(f'Teto global de {self.limite_h}h atingido, encerrando o treino com o que foi aprendido')
            control.should_training_stop = True
        return control


def rodar_experimento_lora(nome_modelo, rotulo, quantizado, epocas, lr=2e-4, max_steps=-1, teto_h=None):
    print(f'=== Experimento {rotulo}: {nome_modelo} | quantizado={quantizado} | épocas={epocas} ===')
    if quantizado:
        config_4bit = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_use_double_quant=True)
        try:
            base = AutoModelForCausalLM.from_pretrained(nome_modelo, quantization_config=config_4bit,
                                                        device_map={'': 0}, dtype=torch.float16)
        except TypeError:
            base = AutoModelForCausalLM.from_pretrained(nome_modelo, quantization_config=config_4bit,
                                                        device_map={'': 0}, torch_dtype=torch.float16)
    else:
        base = carregar_modelo_fp32(nome_modelo)

    print('Avaliando a base ANTES do treino (sem adaptadores)...')
    metricas_antes = calcular_metricas(base, tokenizer, sft_textos_eval, prompts=sft_prompts_eval)
    print(f'Antes: {metricas_antes}')
    respostas_antes = responder_qualitativa(base, tokenizer, avaliacao_qualitativa)

    if quantizado:
        base = prepare_model_for_kbit_training(base)
    else:
        base.enable_input_require_grads()
    base.config.use_cache = False

    config_lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                             bias='none', target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
    modelo = get_peft_model(base, config_lora)
    modelo.print_trainable_parameters()
    try:
        params_treinaveis, params_totais = modelo.get_nb_trainable_parameters()
    except AttributeError:
        params_treinaveis = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
        params_totais = sum(p.numel() for p in modelo.parameters())

    args_experimento = montar_training_args(
        output_dir=f'treino_{rotulo}',
        num_train_epochs=epocas,
        max_steps=max_steps,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        learning_rate=lr,
        weight_decay=0.01,
        warmup_steps=0.05,
        lr_scheduler_type='cosine',
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        fp16=(DEVICE.type == 'cuda' and not quantizado),
        dataloader_num_workers=0,
        logging_steps=20,
        disable_tqdm=True,
        report_to='none',
    )
    callbacks = [TetoDeTempo(teto_h)] if teto_h else []
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    treinador = Trainer(model=modelo, args=args_experimento, train_dataset=sft_train_ds,
                        eval_dataset=sft_eval_ds, data_collator=default_data_collator,
                        callbacks=callbacks)
    resultado = treinador.train()
    tempo_treino = resultado.metrics['train_runtime']
    vram_pico = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == 'cuda' else 0.0
    log_hist = [dict(h) for h in treinador.state.log_history]

    base.config.use_cache = True
    print('Avaliando DEPOIS do treino...')
    metricas_depois = calcular_metricas(modelo, tokenizer, sft_textos_eval, prompts=sft_prompts_eval)
    print(f'Depois: {metricas_depois}')
    respostas_depois = responder_qualitativa(modelo, tokenizer, avaliacao_qualitativa)

    dir_adapter = f'adapter_{rotulo}'
    modelo.save_pretrained(dir_adapter)
    tamanho_adapter_mb = sum(os.path.getsize(os.path.join(raiz, arquivo))
                             for raiz, _, arquivos in os.walk(dir_adapter)
                             for arquivo in arquivos) / 1e6
    remover_checkpoints(f'treino_{rotulo}')

    experimento = {
        'rotulo': rotulo,
        'modelo': nome_modelo,
        'quantizado': quantizado,
        'hiperparametros': {'r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
                            'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                            'learning_rate': lr, 'epocas': epocas, 'micro_batch': 2,
                            'grad_accum': 8, 'max_length': MAX_LEN},
        'params_treinaveis': params_treinaveis,
        'params_totais': params_totais,
        'pct_treinavel': round(params_treinaveis / params_totais * 100, 4),
        'metricas_antes': metricas_antes,
        'metricas_depois': metricas_depois,
        'delta': {
            'entropia_cruzada': round(metricas_depois['entropia_cruzada'] - metricas_antes['entropia_cruzada'], 4),
            'perplexidade': round(metricas_depois['perplexidade'] - metricas_antes['perplexidade'], 2),
            'acuracia_tokens_pp': round(metricas_depois['acuracia_tokens'] - metricas_antes['acuracia_tokens'], 2),
        },
        'treino': {'tempo_s': round(tempo_treino, 1), 'vram_pico_gb': round(vram_pico, 2),
                   'loss_final': round(resultado.training_loss, 4),
                   'tamanho_adaptador_mb': round(tamanho_adapter_mb, 1)},
        'qualitativa_antes': respostas_antes,
        'qualitativa_depois': respostas_depois,
        'log_history': log_hist,
    }
    salvar_json(experimento, f'resultados_exp_{rotulo}.json')
    del treinador, modelo, base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'Experimento {rotulo} concluído: tempo {tempo_treino:.0f}s | VRAM pico {vram_pico:.2f} GB | adapter {tamanho_adapter_mb:.1f} MB')
    return experimento


print('Função de experimento LoRA/QLoRA definida.')

In [ ]:
exp_lora = None
try:
    exp_lora = rodar_experimento_lora(MODEL_NAME, 'lora_qwen05b', quantizado=False, epocas=Q3_EPOCAS,
                                      teto_h=LIMITE_TETO_H)
except Exception:
    traceback.print_exc()
    print('LoRA puro falhou; o notebook segue para tentar o QLoRA.')
liberar_gpu()

In [ ]:
exp_qlora = None
if not BNB_OK:
    print('bitsandbytes indisponível: QLoRA pulado. O LoRA puro cumpre a Questão 3.')
elif tempo_decorrido_h() > 10.5:
    print(f'QLoRA pulado por tempo ({tempo_decorrido_h():.1f}h decorridas). O LoRA puro cumpre a Questão 3.')
else:
    try:
        exp_qlora = rodar_experimento_lora(MODEL_NAME, 'qlora_qwen05b', quantizado=True, epocas=Q3_EPOCAS,
                                           teto_h=LIMITE_TETO_H)
    except Exception:
        traceback.print_exc()
        print('QLoRA falhou. O LoRA puro já cumpre a Questão 3; seguindo em frente.')
liberar_gpu()

### Comparações da Questão 03

Quatro visões: o efeito isolado da quantização 4-bit na base, as curvas de loss das três técnicas, as métricas finais lado a lado e o painel de custo (tempo, VRAM) versus qualidade. O gráfico de parâmetros treináveis usa escala log: o Full SFT atualiza ~494M de parâmetros, o LoRA ~2M.

In [ ]:
experimentos_q3 = [('LoRA', exp_lora, C_LORA), ('QLoRA', exp_qlora, C_QLORA)]
experimentos_q3 = [(rotulo, exp, cor) for rotulo, exp, cor in experimentos_q3 if exp]

if not experimentos_q3:
    print('Nenhum experimento da Questão 3 concluído; comparações puladas.')
else:
    if exp_lora and exp_qlora:
        grafico_metricas_multi(
            [('Base fp32', exp_lora['metricas_antes'], C_BASE), ('Base 4-bit', exp_qlora['metricas_antes'], C_QLORA)],
            'Questão 3: efeito da quantização 4-bit no modelo base (sem treino)', 'q3_quantizacao.png')

    series_loss = [('Full SFT (Q2)', RESULTADOS['q2']['log_history'], C_FULL)]
    for rotulo, exp, cor in experimentos_q3:
        series_loss.append((rotulo, exp['log_history'], cor))
    grafico_curvas_loss(series_loss, 'Questão 3: curvas de loss, Full SFT vs LoRA vs QLoRA', 'q3_curvas_loss.png')

    series_metricas = [('Base (antes)', experimentos_q3[0][1]['metricas_antes'], C_BASE),
                       ('Full SFT', RESULTADOS['q2']['metricas_depois'], C_FULL)]
    for rotulo, exp, cor in experimentos_q3:
        series_metricas.append((rotulo, exp['metricas_depois'], cor))
    grafico_metricas_multi(series_metricas, 'Questão 3: métricas finais por técnica (tokens de resposta)', 'q3_metricas.png')

    itens_params = [('Full SFT (todos os pesos)', RESULTADOS['q2']['treino']['params_treinaveis'])]
    for rotulo, exp, cor in experimentos_q3:
        itens_params.append((f'{rotulo} 0.5B', exp['params_treinaveis']))
    grafico_barh(itens_params, 'Parâmetros treináveis por técnica (escala log)', 'q3_lora_params.png',
                 cor='#4a3aa7', log_x=True)

    entidades = [{'rotulo': 'Full SFT', 'cor': C_FULL,
                  'ppl': RESULTADOS['q2']['metricas_depois']['perplexidade'],
                  'acc': RESULTADOS['q2']['metricas_depois']['acuracia_tokens'],
                  'tempo_min': RESULTADOS['q2']['treino']['tempo_s'] / 60,
                  'vram': RESULTADOS['q2']['treino']['vram_pico_gb']}]
    for rotulo, exp, cor in experimentos_q3:
        entidades.append({'rotulo': rotulo, 'cor': cor,
                          'ppl': exp['metricas_depois']['perplexidade'],
                          'acc': exp['metricas_depois']['acuracia_tokens'],
                          'tempo_min': exp['treino']['tempo_s'] / 60,
                          'vram': exp['treino']['vram_pico_gb']})
    grafico_painel_comparativo(entidades, 'Questão 3: custo e qualidade, Full SFT vs LoRA vs QLoRA',
                               'q3_painel_comparativo.png')

    print(f"{'Técnica':<12} {'PPL depois':>12} {'Acc depois':>12} {'Tempo (min)':>12} {'VRAM (GB)':>10} {'Params trein.':>15}")
    print('-' * 78)
    print(f"{'Full SFT':<12} {RESULTADOS['q2']['metricas_depois']['perplexidade']:>12.2f} {RESULTADOS['q2']['metricas_depois']['acuracia_tokens']:>12.2f} {RESULTADOS['q2']['treino']['tempo_s'] / 60:>12.1f} {RESULTADOS['q2']['treino']['vram_pico_gb']:>10.2f} {RESULTADOS['q2']['treino']['params_treinaveis']:>15,}")
    for rotulo, exp, cor in experimentos_q3:
        print(f"{rotulo:<12} {exp['metricas_depois']['perplexidade']:>12.2f} {exp['metricas_depois']['acuracia_tokens']:>12.2f} {exp['treino']['tempo_s'] / 60:>12.1f} {exp['treino']['vram_pico_gb']:>10.2f} {exp['params_treinaveis']:>15,}")

In [ ]:
if experimentos_q3:
    resultados_lora_q3 = {
        'modelo': MODEL_NAME,
        'dataset': 'vickminari/docentesDC',
        'n_treino': len(sft_pares_treino),
        'n_avaliacao': len(sft_pares_eval),
        'base': {'fp32': exp_lora['metricas_antes'] if exp_lora else None,
                 'int4': exp_qlora['metricas_antes'] if exp_qlora else None},
        'experimentos': {'lora': exp_lora, 'qlora': exp_qlora},
        'comparacao_full_sft': {'metricas_depois': RESULTADOS['q2']['metricas_depois'],
                                'treino': RESULTADOS['q2']['treino']},
    }
    salvar_json(resultados_lora_q3, 'resultados_lora_q3.json')
    RESULTADOS['q3'] = resultados_lora_q3
else:
    print('Sem experimentos concluídos na Questão 3; resultados_lora_q3.json não gerado.')
atualizar_zip_parcial()
status_recursos('fim da Questão 3')

## Seção 4: Bônus, QLoRA do Qwen2.5-1.5B

Cumpre o item opcional do enunciado ("considerar mais de um modelo LLM com parâmetros diferentes"). A célula é totalmente blindada: só roda se o bitsandbytes estiver funcional e se tiverem passado menos de 9,5h de sessão, um callback interrompe o treino se o total passar de 11h e qualquer exceção vira um traceback impresso sem derrubar o notebook. O tokenizer do Qwen2.5-1.5B é o mesmo da família, então os datasets tokenizados são reutilizados.

In [ ]:
exp_bonus = None
if not RODAR_BONUS_15B:
    print('Bônus desativado (RODAR_BONUS_15B=False)')
elif not BNB_OK:
    print('Bônus pulado: bitsandbytes indisponível')
elif tempo_decorrido_h() > LIMITE_BONUS_H:
    print(f'Bônus pulado: {tempo_decorrido_h():.1f}h decorridas (limite {LIMITE_BONUS_H}h)')
else:
    try:
        exp_bonus = rodar_experimento_lora(MODEL_NAME_BONUS, 'qlora_qwen15b', quantizado=True,
                                           epocas=Q3_EPOCAS, max_steps=BONUS_MAX_STEPS,
                                           teto_h=LIMITE_TETO_H)
        salvar_json(exp_bonus, 'resultados_bonus_15b.json')
        RESULTADOS['bonus'] = exp_bonus
    except Exception:
        traceback.print_exc()
        print('Bônus falhou; seguindo para a consolidação com os resultados já salvos.')
    finally:
        liberar_gpu()
        atualizar_zip_parcial()

## Seção 5: Consolidação final

Resumo de todos os experimentos, gráfico geral do trabalho, JSON consolidado e o zip de entrega. Os JSONs pequenos também são impressos no stdout: se algo impedir a publicação dos arquivos de output, os resultados continuam recuperáveis pelo log da versão.

In [ ]:
linhas_resumo = []
if 'q1' in RESULTADOS:
    linhas_resumo.append(('Q1 Pré-treino', RESULTADOS['q1']['metricas_antes'], RESULTADOS['q1']['metricas_depois']))
if 'q2' in RESULTADOS:
    linhas_resumo.append(('Q2 Full SFT', RESULTADOS['q2']['metricas_antes'], RESULTADOS['q2']['metricas_depois']))
if 'q3' in RESULTADOS:
    for rotulo_exp, chave_exp in (('Q3 LoRA', 'lora'), ('Q3 QLoRA', 'qlora')):
        experimento = RESULTADOS['q3']['experimentos'].get(chave_exp)
        if experimento:
            linhas_resumo.append((rotulo_exp, experimento['metricas_antes'], experimento['metricas_depois']))
if 'bonus' in RESULTADOS:
    linhas_resumo.append(('Bônus 1.5B QLoRA', RESULTADOS['bonus']['metricas_antes'],
                          RESULTADOS['bonus']['metricas_depois']))

print(f"{'Experimento':<20} {'PPL antes':>10} {'PPL depois':>11} {'EC antes':>9} {'EC depois':>10} {'Acc antes':>10} {'Acc depois':>11}")
print('-' * 88)
for nome, antes, depois in linhas_resumo:
    print(f"{nome:<20} {antes['perplexidade']:>10.2f} {depois['perplexidade']:>11.2f} {antes['entropia_cruzada']:>9.4f} {depois['entropia_cruzada']:>10.4f} {antes['acuracia_tokens']:>10.2f} {depois['acuracia_tokens']:>11.2f}")

if linhas_resumo:
    nomes = [linha[0] for linha in linhas_resumo]
    ppl_antes = [linha[1]['perplexidade'] for linha in linhas_resumo]
    ppl_depois = [linha[2]['perplexidade'] for linha in linhas_resumo]
    acc_antes = [linha[1]['acuracia_tokens'] for linha in linhas_resumo]
    acc_depois = [linha[2]['acuracia_tokens'] for linha in linhas_resumo]
    reducao_ppl = [round((1 - d / a) * 100, 1) if a > 0 else 0 for a, d in zip(ppl_antes, ppl_depois)]
    posicoes = np.arange(len(nomes))
    fig, eixos = plt.subplots(1, 3, figsize=(16, 4.6))
    eixos[0].bar(posicoes - 0.19, ppl_antes, width=0.38, color=COR_ANTES, label='Antes')
    eixos[0].bar(posicoes + 0.19, ppl_depois, width=0.38, color=COR_DEPOIS, label='Depois')
    eixos[0].set_yscale('log')
    eixos[0].set_title('Perplexidade (escala log)')
    eixos[0].legend(frameon=False)
    eixos[1].bar(posicoes - 0.19, acc_antes, width=0.38, color=COR_ANTES, label='Antes')
    eixos[1].bar(posicoes + 0.19, acc_depois, width=0.38, color=COR_DEPOIS, label='Depois')
    eixos[1].set_title('Acurácia de Tokens (%)')
    eixos[1].legend(frameon=False)
    barras_reducao = eixos[2].bar(posicoes, reducao_ppl, width=0.55, color=C_LORA)
    rotular_barras(eixos[2], barras_reducao, sufixo='%', casas=1)
    eixos[2].set_title('Redução relativa da Perplexidade')
    for ax in eixos:
        ax.set_xticks(posicoes)
        ax.set_xticklabels(nomes, rotation=18, ha='right')
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.suptitle('Resumo geral do trabalho: antes vs depois em cada experimento', fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, 'resumo_geral.png')

if 'q2' in RESULTADOS and 'q3' in RESULTADOS and 'bonus' in RESULTADOS:
    itens_params = [('Full SFT (todos os pesos)', RESULTADOS['q2']['treino']['params_treinaveis'])]
    for rotulo_exp, chave_exp in (('LoRA 0.5B', 'lora'), ('QLoRA 0.5B', 'qlora')):
        experimento = RESULTADOS['q3']['experimentos'].get(chave_exp)
        if experimento:
            itens_params.append((rotulo_exp, experimento['params_treinaveis']))
    itens_params.append(('QLoRA 1.5B (bônus)', RESULTADOS['bonus']['params_treinaveis']))
    grafico_barh(itens_params, 'Parâmetros treináveis por técnica (escala log)', 'q3_lora_params.png',
                 cor='#4a3aa7', log_x=True)

In [ ]:
resultados_consolidados = {
    'modo_teste': MODO_TESTE,
    'tempo_total_h': round(tempo_decorrido_h(), 2),
    'resumo': [{'experimento': nome, 'antes': antes, 'depois': depois}
               for nome, antes, depois in linhas_resumo],
}
salvar_json(resultados_consolidados, 'resultados_consolidados.json')

ARQUIVOS_BACKUP_LOG = ['resultados_metricas.json', 'resultados_benchmark_base.json',
                       'resultados_benchmark_treinado.json', 'resultados_sft_q2.json',
                       'resultados_lora_q3.json', 'resultados_bonus_15b.json',
                       'resultados_consolidados.json']
for nome in ARQUIVOS_BACKUP_LOG:
    if os.path.exists(nome) and os.path.getsize(nome) < 400000:
        print(f'===== {nome} =====')
        with open(nome, encoding='utf-8') as arquivo:
            print(arquivo.read())

os.makedirs('entrega', exist_ok=True)
for nome in os.listdir('.'):
    if nome.endswith('.json'):
        shutil.copy(nome, os.path.join('entrega', nome))
    elif nome.startswith('adapter_') and os.path.isdir(nome):
        shutil.copytree(nome, os.path.join('entrega', nome), dirs_exist_ok=True)
if os.path.isdir(DIR_FIGURAS):
    shutil.copytree(DIR_FIGURAS, os.path.join('entrega', DIR_FIGURAS), dirs_exist_ok=True)
shutil.make_archive('resultados_completos', 'zip', 'entrega')
print('Zip final gerado: resultados_completos.zip (JSONs + figuras + adapters)')
print('Os modelos completos ficam nas pastas q1_modelo_final/ e q2_modelo_final/ do output.')
status_recursos('fim do notebook')
print(f'Notebook concluído em {tempo_decorrido_h():.2f}h')